# SpringGPT 1 Mini — LFM3 text and images

Enable Internet and select **GPU T4 x2** in Session options, then Run All. This loads LFM3 1.7B for text on one GPU and LFM3-VL 2B for images on the other. A small bounded queue shares each GPU. Keep the last cell running.

The temporary tunnel is public while running. Do not send private information. Kaggle session time and quota limits apply.


In [ ]:
!pip -q install "transformers==5.22.1" "fastapi>=0.115,<1" "uvicorn>=0.30,<1" pillow


In [ ]:
%%writefile springgpt_server.py
"""SpringGPT 1 Mini: LFM3 text inference on up to two Kaggle GPUs."""
import asyncio
import base64
import io
import logging
import os
from contextlib import asynccontextmanager
from typing import Literal

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

MODEL = "LiquidAI/LFM3-1.7B"
REVISION = "ac2586a34f1a1c03329dbdcdefabd8b6478ddaa7"
SYSTEM_PROMPT = ("You are SpringGPT 1 Mini, an AI assistant made by SpropenAI (by WeirdMM). "
                 "SpropenAI is a parody name. Be helpful, accurate, and concise. "
                 "When asked who you are or who made you, say: I am SpringGPT 1 Mini, made by SpropenAI (by WeirdMM). "
                 "If you are unsure, say so.")
VISION_MODEL = "LiquidAI/LFM3-VL-2B"
VISION_REVISION = "ba44bd51db20a2753f28c273a1571ccaa210d954"
MAX_NEW_TOKENS = int(os.environ.get("SPRINGGPT_MAX_NEW_TOKENS", "1024"))
QUEUE_LIMIT = 12
QUEUE_TIMEOUT = 40
text_slots = asyncio.Queue()
vision_slots = asyncio.Queue()
waiting = 0
ready = False

class Message(BaseModel):
    role: Literal["user", "assistant"]
    content: str = Field(min_length=1, max_length=4000)

class Chat(BaseModel):
    messages: list[Message] = Field(min_length=1, max_length=20)
    image_base64: str | None = Field(default=None, max_length=2800000)

@asynccontextmanager
async def lifespan(app):
    global ready
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForImageTextToText, AutoProcessor
    count = torch.cuda.device_count()
    if count < 2:
        raise RuntimeError("Two GPUs are required for simultaneous text and image service")
    tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, trust_remote_code=True)
    tokenizer.truncation_side = "left"
    text_model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION,
                    trust_remote_code=True, dtype=torch.float16).to("cuda:0").eval()
    text_slots.put_nowait((tokenizer, text_model))
    print("SpringGPT Mini text ready on cuda:0", flush=True)
    processor = AutoProcessor.from_pretrained(VISION_MODEL, revision=VISION_REVISION, trust_remote_code=True)
    vision_model = AutoModelForImageTextToText.from_pretrained(VISION_MODEL, revision=VISION_REVISION,
                    trust_remote_code=True, dtype=torch.float16).to("cuda:1").eval()
    vision_slots.put_nowait((processor, vision_model))
    print("SpringGPT image understanding ready on cuda:1", flush=True)
    ready = True
    yield
    ready = False

app = FastAPI(lifespan=lifespan)

@app.get("/health")
def health():
    return {"ready": ready, "model": "SpringGPT 1 Mini", "capabilities": {"images": True}}

def generate(slot, messages, image_base64):
    import torch
    from PIL import Image
    processor, model = slot
    conversation = [{"role":"system", "content":SYSTEM_PROMPT}] + messages
    if image_base64:
        raw = base64.b64decode(image_base64, validate=True)
        if len(raw) > 2 * 1024 * 1024:
            raise ValueError("Image too large")
        Image.MAX_IMAGE_PIXELS = 20_000_000
        image = Image.open(io.BytesIO(raw))
        if image.format not in ("JPEG", "PNG"):
            raise ValueError("Unsupported image format")
        image = image.convert("RGB")
        image.thumbnail((1024, 1024))
        # The last user message supplies the image; older messages remain text only.
        for message in reversed(conversation):
            if message["role"] == "user":
                message["content"] = [{"type":"image", "image":image}, {"type":"text", "text":message["content"]}]
                break
        encoded = processor.apply_chat_template(conversation, add_generation_prompt=True,
                    tokenize=True, return_dict=True, return_tensors="pt", enable_thinking=True).to(model.device, dtype=torch.float16)
        tokenizer = processor.tokenizer
    else:
        tokenizer = processor
        prompt = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True, enable_thinking=True)
        encoded = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(model.device)
    with torch.inference_mode():
        result = model.generate(**encoded, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                    temperature=0.1, top_k=50, repetition_penalty=1.1,
                    pad_token_id=tokenizer.pad_token_id)
    output = result[0][encoded.input_ids.shape[1]:].tolist()
    decoded = tokenizer.decode(output, skip_special_tokens=False)
    if "</think>" not in decoded:
        raise ValueError("Thinking did not finish before the output limit")
    final = decoded.rsplit("</think>", 1)[1]
    for token in tokenizer.all_special_tokens:
        final = final.replace(token, "")
    final = final.strip()
    if not final:
        raise ValueError("No final answer")
    return final

@app.post("/chat")
async def chat(request: Chat):
    global waiting
    if not ready:
        raise HTTPException(status_code=503, detail="Model is starting")
    if waiting >= QUEUE_LIMIT:
        raise HTTPException(status_code=429, detail="Service is busy")
    chosen_slots = vision_slots if request.image_base64 else text_slots
    waiting += 1
    try:
        try:
            slot = await asyncio.wait_for(chosen_slots.get(), timeout=QUEUE_TIMEOUT)
        except asyncio.TimeoutError:
            raise HTTPException(status_code=429, detail="Service is busy")
    finally:
        waiting -= 1
    # A cancelled request must not release a GPU while its thread is still running.
    task = asyncio.create_task(asyncio.to_thread(generate, slot, [m.model_dump() for m in request.messages], request.image_base64))
    def return_slot(_):
        chosen_slots.put_nowait(slot)
    task.add_done_callback(return_slot)
    try:
        result = await asyncio.shield(task)
    except asyncio.CancelledError:
        raise
    except Exception:
        logging.exception("Generation failed")
        raise HTTPException(status_code=503, detail="The model could not finish its reply")
    return {"reply": result}


In [ ]:
import os, urllib.request
url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
urllib.request.urlretrieve(url, "/kaggle/working/cloudflared")
os.chmod("/kaggle/working/cloudflared", 0o755)
print("Tunnel client ready.")


In [ ]:
import subprocess, threading, re, time, sys

server = subprocess.Popen([sys.executable, "-m", "uvicorn", "springgpt_server:app", "--host", "127.0.0.1", "--port", "8000"])
tunnel = subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"], stderr=subprocess.PIPE, text=True, bufsize=1)

def show_tunnel():
    for line in tunnel.stderr:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            print("\nSpringGPT backend URL:", match.group(0), flush=True)
            print("Set backend_url to this value and enabled to true in the GitHub config file. Wait for the model to finish loading.", flush=True)

threading.Thread(target=show_tunnel, daemon=True).start()
print("Starting model and tunnel; keep this cell running.", flush=True)
try:
    while server.poll() is None and tunnel.poll() is None:
        time.sleep(2)
finally:
    server.terminate()
    tunnel.terminate()
    print("Server stopped.")
